<a href="https://colab.research.google.com/github/P-shettar/ML-LAB-02/blob/main/ML_lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install wittgenstein

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wittgenstein: filename=wittgenstein-0.3.5-py3-none-any.whl size=99081 sha256=7fe12cd9602bd3483a7ed2f70cdafbabe623ed5fea2e58339bebd74ed65cc7d2
  Stored in directory: /root/.cache/pip/wheels/03/8e/f7/4afc64184996e67ef74ebefa7d74b324534a079fc3462242aa
Successfully built wittgenstein


In [8]:
!pip install -q wittgenstein

import pandas as pd
import wittgenstein as lw
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

data = pd.read_csv("weather.csv")
data["Rain"] = data["Data.Precipitation"].apply(lambda x: "Yes" if x > 0 else "No")

def bin3(col):
    return pd.cut(col, bins=3, labels=["Low", "Medium", "High"]).astype(str)

df = pd.DataFrame({
    "AvgTemp":   bin3(data["Data.Temperature.Avg Temp"]),
    "MaxTemp":   bin3(data["Data.Temperature.Max Temp"]),
    "MinTemp":   bin3(data["Data.Temperature.Min Temp"]),
    "WindSpeed": bin3(data["Data.Wind.Speed"]),
    "Rain":      data["Rain"],
}).dropna()
df = df[~df.eq("nan").any(axis=1)]   # astype(str) turns NaN into "nan"

print(df["Rain"].value_counts(normalize=True))   # check class balance

train, test = train_test_split(df, test_size=0.3, random_state=42, stratify=df["Rain"])

model = lw.RIPPER(random_state=42)
model.fit(train, class_feat="Rain", pos_class="Yes")

print("\n========== RIPPER RULES ==========")
print(model.ruleset_)

# predict() returns booleans -> convert back to labels
X_test = test.drop(columns="Rain")
pred = ["Yes" if p else "No" for p in model.predict(X_test)]
y_test = test["Rain"]

print("\nAccuracy: ", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, pos_label="Yes", zero_division=0))
print("Recall:   ", recall_score(y_test, pred, pos_label="Yes", zero_division=0))
print(classification_report(y_test, pred, zero_division=0))

Rain
Yes    0.741743
No     0.258257
Name: proportion, dtype: float64

========== RIPPER RULES ==========
[[AvgTemp=Medium^MaxTemp=Medium^MinTemp=High] V [AvgTemp=Medium^MaxTemp=Medium] V [MinTemp=High^MaxTemp=High] V [MaxTemp=Low^MinTemp=Medium^AvgTemp=Medium] V [AvgTemp=Low] V [MinTemp=Medium^MaxTemp=Medium]]

Accuracy:  0.7356161656380649
Precision: 0.7694382022471911
Recall:    0.918947933440687
              precision    recall  f1-score   support

          No       0.47      0.21      0.29      1297
         Yes       0.77      0.92      0.84      3726

    accuracy                           0.74      5023
   macro avg       0.62      0.56      0.56      5023
weighted avg       0.69      0.74      0.70      5023

